# Experimento 02: version limpia de cajas heuristicas

**Estado:** rama experimental conservada para trazabilidad, no estrategia final de entrega.

**Deriva de:** 01_cajas_heuristicas_curva_dinamica -> limpieza rapida previa a la red neuronal.

**Por que se conserva:** Se conserva como cierre de la etapa heuristica antes de pasar a una detectora entrenable.

Este notebook se deja con salidas limpias para que GitHub sea liviano. La estrategia principal queda en `notebooks/`; esta carpeta explica los caminos que se probaron y por que no todos terminaron como modelo final.

<!-- codex-experimento-conservado -->


# Vertebra semana final

Notebook limpio para la etapa final de deteccion automatica de vertebras.

Objetivo:
- Mantener solo la logica necesaria para cargar datos, estimar curvas, generar cajas, evaluar deteccion y probar MedSAM basico.
- Usar lo aprendido: normales con `bordes` / `central_robusto`; escoliosis con `ruta_dinamica` / multi-ruta cercana.
- Corregir desplazamientos verticales por tramos, no solo en L5.

## 1. Configuracion base
Rutas, clases y artefactos ya exportados para MedSAM.

In [ ]:
import os, json, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
print('âœ… Entorno listo')

# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# DATASET ACTIVO
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
DATASET_ROOT = Path('C:/Users/luisf/Downloads/ProyectoFinal/Scoliosis_Dataset')

# Subcarpetas fijas del dataset limpio
DIR_NORMAL      = DATASET_ROOT / 'Normal'
DIR_SCOLIOSIS   = DATASET_ROOT / 'Scoliosis'
DIR_MASK_ID     = DATASET_ROOT / 'LabelMultiClass_ID_PNG'
DIR_MASK_BIN    = DATASET_ROOT / 'LabelBinaryJPG'
DIR_METRICS     = DATASET_ROOT / 'RadiographMetrics'
COBB_SUMMARY_PATH = DIR_METRICS / 'metricas_cobb_resumen_recalculado.csv'
DATASET_INDEX_PATH = DATASET_ROOT / 'indice_dataset.csv'
LABELS_DICT_PATH = DATASET_ROOT / 'diccionario_etiquetas_T1_T12_L1_L5.json'

# Salida separada para no mezclarla con exportaciones del dataset anterior
OUTPUT_ROOT = DATASET_ROOT.parent / 'dataset_procesado_scoliosis_medsam'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Raiz MedSAM correcta para esta version reducida.
# Se define explicitamente para evitar que la busqueda automatica tome dataset_procesado/medsam viejo.
OUT_MEDSAM = OUTPUT_ROOT / 'medsam'

# Cargar diccionario oficial de la versiÃ³n limpia T1â€“T12/L1â€“L5
with open(LABELS_DICT_PATH, encoding='utf-8') as f:
    LABELS_DICT = json.load(f)

MAPEO_ID = {int(k): v for k, v in LABELS_DICT['mascara_multiclase_id_png'].items()}
CLASES = {k: v for k, v in MAPEO_ID.items() if 1 <= k <= 17}
N_CLASES = len(CLASES)
NOMBRES_CLASES = [CLASES[i] for i in range(1, N_CLASES + 1)]

print(f'Dataset:  {DATASET_ROOT}')
print(f'Existe:   {DATASET_ROOT.exists()}')
print(f'Indice:   {DATASET_INDEX_PATH.exists()}')
print(f'COCO:     no encontrado en esta version')
print(f'Clases:   {N_CLASES} â†’ {NOMBRES_CLASES}')


In [ ]:
# ==========================================
# 2) CLASES DEL DATASET Y CLASES OBJETIVO
# ==========================================

# La version limpia ya usa IDs consecutivos:
# 0 = fondo, 1..12 = T1..T12, 13..17 = L1..L5.
CLASES_OBJETIVO = NOMBRES_CLASES.copy()
N_CLASES = len(CLASES_OBJETIVO)

VERTEBRA_TO_ID = {nombre: idx for idx, nombre in MAPEO_ID.items() if idx != 0}
ID_TO_VERTEBRA = {idx: nombre for nombre, idx in VERTEBRA_TO_ID.items()}

# En esta version el ID real de la mascara y el ID local de evaluacion son iguales.
CLASS_TO_ID = VERTEBRA_TO_ID.copy()
ID_TO_CLASS = {idx: nombre for nombre, idx in CLASS_TO_ID.items()}

print("=" * 80)
print("Configuracion actual para MEDSAM")
print("=" * 80)
print("Dataset activo: Scoliosis_Dataset limpio")
print("Clases objetivo (17):", CLASES_OBJETIVO)
print("IDs de mascara para T1-L5:")
for c in CLASES_OBJETIVO:
    print(f"  {c:>3} -> {VERTEBRA_TO_ID[c]}")
print("=" * 80)

# ==========================================
# 3) FUNCIONES AUXILIARES PARA BUSCAR ARCHIVOS
# ==========================================

def buscar_archivo(nombre_archivo, raiz_inicial=Path("."), max_resultados=20):
    encontrados = []
    for p in raiz_inicial.rglob(nombre_archivo):
        encontrados.append(p)
        if len(encontrados) >= max_resultados:
            break
    return encontrados

EXTS_IMG = [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]

def buscar_archivo_por_stem(carpeta, stem):
    for ext in EXTS_IMG:
        p = carpeta / f"{stem}{ext}"
        if p.exists():
            return p
    return None


In [ ]:
# ==========================================
# 4) DETECTAR LA RAIZ DE MEDSAM
# ==========================================

MEDSAM_DATA_ROOT = None

variables_candidatas = [
    "MEDSAM_DATA_ROOT",
    "medsam_data_root",
    "MEDSAM_ROOT",
    "medsam_root",
    "OUT_MEDSAM",
    "out_medsam",
    "EXPORT_MEDSAM_DIR",
    "export_medsam_dir"
]

for var_name in variables_candidatas:
    if var_name in globals():
        try:
            MEDSAM_DATA_ROOT = Path(globals()[var_name])
            if MEDSAM_DATA_ROOT.exists():
                print(f"Se usara {var_name} = {MEDSAM_DATA_ROOT}")
                break
        except Exception:
            pass

if MEDSAM_DATA_ROOT is None:
    candidatos_raiz = []

    if "OUTPUT_ROOT" in globals():
        candidatos_raiz.append(Path(OUTPUT_ROOT) / "medsam")

    if "DATASET_ROOT" in globals():
        candidatos_raiz.append(DATASET_ROOT.parent / "dataset_procesado_scoliosis_medsam" / "medsam")

    candidatos_raiz.extend([
        Path("C:/Users/luisf/Downloads/ProyectoFinal/dataset_procesado_scoliosis_medsam/medsam"),
        Path("dataset_procesado_scoliosis_medsam/medsam"),
    ])

    for candidato in candidatos_raiz:
        candidato = Path(candidato)
        if (candidato / "train" / "prompts.json").exists() and (candidato / "val" / "prompts.json").exists():
            MEDSAM_DATA_ROOT = candidato
            print(f"MEDSAM_DATA_ROOT definido por ruta limpia: {MEDSAM_DATA_ROOT}")
            break

if MEDSAM_DATA_ROOT is None:
    prompts_encontrados = buscar_archivo("prompts.json", Path("."))
    prompts_medsam = [p for p in prompts_encontrados if "dataset_procesado_scoliosis_medsam" in str(p).lower()]

    if len(prompts_medsam) == 0:
        prompts_medsam = [p for p in prompts_encontrados if "medsam" in str(p).lower()]

    if len(prompts_medsam) > 0:
        MEDSAM_DATA_ROOT = prompts_medsam[0].parent.parent
        print(f"MEDSAM_DATA_ROOT detectado automaticamente: {MEDSAM_DATA_ROOT}")
    elif len(prompts_encontrados) > 0:
        MEDSAM_DATA_ROOT = prompts_encontrados[0].parent.parent
        print(f"MEDSAM_DATA_ROOT detectado automaticamente: {MEDSAM_DATA_ROOT}")
    else:
        raise FileNotFoundError(
            "No se encontro prompts.json. "
            "Corre primero la exportacion a MEDSAM o indica el path exacto."
        )

MEDSAM_DATA_ROOT = Path(MEDSAM_DATA_ROOT)
print("MEDSAM_DATA_ROOT final:", MEDSAM_DATA_ROOT)

# ==========================================
# 5) RESOLVER SPLITS Y PATHS
# ==========================================

SPLITS = ["train", "val", "test"]

def resolver_paths_split(split):
    split_dir = MEDSAM_DATA_ROOT / split
    if not split_dir.exists():
        raise FileNotFoundError(f"No existe el split: {split_dir}")

    prompts_path = split_dir / "prompts.json"
    if not prompts_path.exists():
        raise FileNotFoundError(f"No existe prompts.json en: {split_dir}")

    candidatos_img = ["images", "imgs", "image", "jpg", "png"]
    candidatos_mask = ["masks", "labels", "mask", "annotations"]

    image_dir = None
    mask_dir = None

    for c in candidatos_img:
        p = split_dir / c
        if p.exists() and p.is_dir():
            image_dir = p
            break

    for c in candidatos_mask:
        p = split_dir / c
        if p.exists() and p.is_dir():
            mask_dir = p
            break

    if image_dir is None:
        raise FileNotFoundError(f"No encontre carpeta de imagenes dentro de {split_dir}")

    if mask_dir is None:
        raise FileNotFoundError(f"No encontre carpeta de mascaras dentro de {split_dir}")

    return split_dir, image_dir, mask_dir, prompts_path

SPLIT_INFO = {}
for split in SPLITS:
    split_dir, image_dir, mask_dir, prompts_path = resolver_paths_split(split)
    SPLIT_INFO[split] = {
        "split_dir": split_dir,
        "image_dir": image_dir,
        "mask_dir": mask_dir,
        "prompts_path": prompts_path
    }

print("\nResumen de paths detectados:")
for split in SPLITS:
    print(f"\n[{split}]")
    print(" split_dir   :", SPLIT_INFO[split]["split_dir"])
    print(" image_dir   :", SPLIT_INFO[split]["image_dir"])
    print(" mask_dir    :", SPLIT_INFO[split]["mask_dir"])
    print(" prompts_path:", SPLIT_INFO[split]["prompts_path"])

# ==========================================
# 6) CARGAR PROMPTS Y REVISAR SU ESTRUCTURA
# ==========================================

PROMPTS = {}
for split in SPLITS:
    with open(SPLIT_INFO[split]["prompts_path"], "r", encoding="utf-8") as f:
        PROMPTS[split] = json.load(f)

print("\nCantidad de entradas en prompts.json:")
for split in SPLITS:
    print(f"  {split}: {len(PROMPTS[split])}")

for split in SPLITS:
    print("\n" + "=" * 80)
    print(f"SPLIT: {split}")
    print("type(PROMPTS[split]):", type(PROMPTS[split]))

    if isinstance(PROMPTS[split], list):
        print("len:", len(PROMPTS[split]))
        if len(PROMPTS[split]) > 0:
            print("type primer elemento:", type(PROMPTS[split][0]))
            if isinstance(PROMPTS[split][0], dict):
                print("keys primer elemento:", list(PROMPTS[split][0].keys()))
                print("primer elemento completo:")
                print(PROMPTS[split][0])


In [ ]:
# ==========================================
# 7) FUNCIONES DE CARGA, FILTRADO Y MASCARAS
# ==========================================

def cargar_imagen(path_imagen):
    img = Image.open(path_imagen).convert("RGB")
    return np.array(img)

def cargar_mascara(path_mascara):
    mask = Image.open(path_mascara)
    return np.array(mask).astype(np.int32)

def normalizar_nombre_vertebra(nombre):
    if nombre is None:
        return None
    nombre = str(nombre).strip().upper()
    return nombre if nombre in CLASES_OBJETIVO else None

def filtrar_prompts_t1_l5(prompts_split):
    # Nota metodologica: los prompts se leen del archivo exportado; en esta version vienen de las mascaras GT.
    """
    Estructura real:
    [
        {
            "patient_id": "...",
            "prompts": {
                "1": {"vertebra": "T1", "bbox_xyxy": [...]},
                ...
            }
        },
        ...
    ]
    """
    prompts_filtrados = {}

    if not isinstance(prompts_split, list):
        raise TypeError(f"Se esperaba list en prompts_split y llegÃ³ {type(prompts_split)}")

    for item in prompts_split:
        if not isinstance(item, dict):
            continue

        patient_id = item.get("patient_id")
        prompts_item = item.get("prompts")

        if patient_id is None or prompts_item is None:
            continue

        sub = {}

        if isinstance(prompts_item, dict):
            for _, info in prompts_item.items():
                if not isinstance(info, dict):
                    continue

                nombre = normalizar_nombre_vertebra(info.get("vertebra"))
                if nombre is not None:
                    sub[nombre] = info

        elif isinstance(prompts_item, list):
            for info in prompts_item:
                if not isinstance(info, dict):
                    continue

                nombre = normalizar_nombre_vertebra(info.get("vertebra"))
                if nombre is not None:
                    sub[nombre] = info

        if len(sub) > 0:
            prompts_filtrados[patient_id] = sub

    return prompts_filtrados

def construir_mask_binaria_vertebra(mask_multiclase, vertebra_objetivo):
    """
    Convierte la mascara multiclase global en una mascara binaria para una sola vertebra.
    """
    if vertebra_objetivo not in VERTEBRA_TO_ID:
        raise ValueError(f"Vertebra no conocida: {vertebra_objetivo}")

    vertebra_id_real = VERTEBRA_TO_ID[vertebra_objetivo]
    mask_bin = (mask_multiclase == vertebra_id_real).astype(np.uint8)
    return mask_bin

def construir_mask_multiclase_t1_l5(mask_multiclase):
    """
    Devuelve la mascara T1-L5 en IDs locales 1..17:
    0 = fondo
    1..17 = T1..L5
    """
    # En Scoliosis_Dataset las mascaras ya vienen en 0..17.
    # Se fuerza uint8 y se eliminan posibles IDs fuera del diccionario por seguridad.
    nueva = mask_multiclase.astype(np.uint8).copy()
    nueva[~np.isin(nueva, list(range(0, N_CLASES + 1)))] = 0
    return nueva

def extraer_bbox_desde_mask(mask_binaria):
    ys, xs = np.where(mask_binaria > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None
    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()
    return [int(x0), int(y0), int(x1), int(y1)]

def resolver_paths_muestra(split, patient_id):
    image_dir = SPLIT_INFO[split]["image_dir"]
    mask_dir = SPLIT_INFO[split]["mask_dir"]

    path_imagen = buscar_archivo_por_stem(image_dir, patient_id)
    path_mascara = buscar_archivo_por_stem(mask_dir, patient_id)

    if path_imagen is None:
        raise FileNotFoundError(f"No encontre imagen para {patient_id} en {image_dir}")

    if path_mascara is None:
        raise FileNotFoundError(f"No encontre mascara para {patient_id} en {mask_dir}")

    return path_imagen, path_mascara

# ==========================================
# 8) FILTRAR T1-L5 Y TOMAR UNA MUESTRA DE PRUEBA
# ==========================================

PROMPTS_FILTRADOS = {
    split: filtrar_prompts_t1_l5(PROMPTS[split])
    for split in SPLITS
}

print("\nCantidad de imagenes con prompts T1-L5:")
for split in SPLITS:
    print(f"  {split}: {len(PROMPTS_FILTRADOS[split])}")

for split in SPLITS:
    print("\n" + "=" * 80)
    print(f"SPLIT: {split}")
    print("imagenes filtradas:", len(PROMPTS_FILTRADOS[split]))

    if len(PROMPTS_FILTRADOS[split]) > 0:
        sample_key = next(iter(PROMPTS_FILTRADOS[split]))
        print("sample_key:", sample_key)
        print("vertebras disponibles:", list(PROMPTS_FILTRADOS[split][sample_key].keys()))
        primera_vertebra = next(iter(PROMPTS_FILTRADOS[split][sample_key]))
        print("ejemplo prompt:")
        print(PROMPTS_FILTRADOS[split][sample_key][primera_vertebra])

split = "train"

if len(PROMPTS_FILTRADOS[split]) == 0:
    raise RuntimeError("No se encontraron prompts T1-L5 en train.")

sample_key = next(iter(PROMPTS_FILTRADOS[split].keys()))
path_imagen, path_mascara = resolver_paths_muestra(split, sample_key)

imagen = cargar_imagen(path_imagen)
mascara = cargar_mascara(path_mascara)

# mascara remapeada solo para visualizacion T1-L5
mascara_t1_l5 = construir_mask_multiclase_t1_l5(mascara)

prompts_sample = PROMPTS_FILTRADOS[split][sample_key]

print("\nMuestra seleccionada:")
print(" split       :", split)
print(" sample_key  :", sample_key)
print(" path_imagen :", path_imagen)
print(" path_mascara:", path_mascara)
print(" vertebras en prompts:", list(prompts_sample.keys()))
print(" shape imagen:", imagen.shape)
print(" shape mask original :", mascara.shape)
print(" unique mask original:", np.unique(mascara)[:30])
print(" unique mask T1-L5   :", np.unique(mascara_t1_l5)[:30])

fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(imagen)
ax[0].set_title("Imagen")
ax[0].axis("off")

ax[1].imshow(mascara_t1_l5, cmap="nipy_spectral")
ax[1].set_title("Mascara remapeada T1-L5")
ax[1].axis("off")

plt.tight_layout()
plt.show()


## MedSAM opcional y metricas

In [ ]:
# ============================================================
# MedSAM opcional
# ============================================================

EJECUTAR_CARGA_MEDSAM = False
REQUERIR_CUDA_PARA_MEDSAM = True

if EJECUTAR_CARGA_MEDSAM:
    import torch
    from segment_anything import sam_model_registry
    from segment_anything import SamPredictor

    MedSAM_CKPT_PATH = "C:/Users/luisf/MedSAM/work_dir/MedSAM/medsam_vit_b.pth"

    if REQUERIR_CUDA_PARA_MEDSAM and not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA no esta disponible. MedSAM en CPU puede tardar demasiado; "
            "activa un entorno con GPU o cambia REQUERIR_CUDA_PARA_MEDSAM=False si aceptas CPU."
        )

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Using device:", device)
    if device == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))
        torch.backends.cudnn.benchmark = True

    medsam_model = sam_model_registry["vit_b"](checkpoint=MedSAM_CKPT_PATH)
    medsam_model = medsam_model.to(device)
    medsam_model.eval()
    predictor = SamPredictor(medsam_model)
    print("MedSAM cargado correctamente")
else:
    predictor = None
    print("MedSAM no cargado. Cambia EJECUTAR_CARGA_MEDSAM=True para probar segmentacion basica con CUDA.")

In [ ]:
# ==========================================
# 24) FUNCIONES DE EVALUACION GLOBAL
# ==========================================

def expand_bbox_xyxy_pequena(bbox, img_shape, frac_x=0.04, frac_y=0.06):
    x0, y0, x1, y1 = bbox
    h, w = img_shape[:2]

    bw = x1 - x0
    bh = y1 - y0

    pad_x = int(round(bw * frac_x))
    pad_y = int(round(bh * frac_y))

    x0n = max(0, x0 - pad_x)
    y0n = max(0, y0 - pad_y)
    x1n = min(w - 1, x1 + pad_x)
    y1n = min(h - 1, y1 + pad_y)

    return [x0n, y0n, x1n, y1n]

def evaluar_pred(mask_pred, mask_gt, eps=1e-8):
    inter = np.logical_and(mask_pred == 1, mask_gt == 1).sum()
    union = np.logical_or(mask_pred == 1, mask_gt == 1).sum()
    dice = (2 * inter + eps) / (mask_pred.sum() + mask_gt.sum() + eps)
    iou = (inter + eps) / (union + eps)
    return float(dice), float(iou)

# ==========================================
# 37) VERSION CON MAPA DE SCORE PARA SOLAPAMIENTOS
# ==========================================

def reconstruir_mascara_semantica_medsam_con_score(
    imagen,
    mascara_gt_multiclase,
    prompts_sample,
    predictor,
    frac_x=0.04,
    frac_y=0.06,
    return_quality=False
):
    predictor.set_image(imagen)

    h, w = imagen.shape[:2]
    mask_sem_pred = np.zeros((h, w), dtype=np.uint8)
    score_map = np.zeros((h, w), dtype=np.float32)
    pred_count = np.zeros((h, w), dtype=np.uint8)
    mask_sem_gt = construir_mask_multiclase_t1_l5(mascara_gt_multiclase)

    detalles = []

    for vertebra_objetivo in CLASES_OBJETIVO:
        if vertebra_objetivo not in prompts_sample:
            continue

        # Cada bbox usada aqui proviene del prompt exportado. Si el prompt fue creado desde GT,
        # la metrica evalua segmentacion condicionada por una caja ideal, no deteccion automatica.
        prompt_info = prompts_sample[vertebra_objetivo]
        bbox_original = prompt_info["bbox_xyxy"]
        bbox_expandida = expand_bbox_xyxy_pequena(
            bbox_original,
            imagen.shape,
            frac_x=frac_x,
            frac_y=frac_y
        )

        box_np = np.array(bbox_expandida, dtype=np.float32)[None, :]

        masks, scores, logits = predictor.predict(
            box=box_np,
            multimask_output=False
        )

        mask_pred_bin = masks[0].astype(np.uint8)
        score_pred = float(scores[0])
        pred_count += mask_pred_bin

        id_local = CLASS_TO_ID[vertebra_objetivo]

        # En zonas solapadas se conserva la vertebra con mayor score interno de MedSAM.
        # Este score no es Dice/IoU; solo se usa como regla de desempate entre predicciones.
        update_idx = (mask_pred_bin == 1) & (score_pred > score_map)
        mask_sem_pred[update_idx] = id_local
        score_map[update_idx] = score_pred

        mask_gt_bin = construir_mask_binaria_vertebra(mascara_gt_multiclase, vertebra_objetivo)

        inter = np.logical_and(mask_pred_bin == 1, mask_gt_bin == 1).sum()
        union = np.logical_or(mask_pred_bin == 1, mask_gt_bin == 1).sum()

        dice_v = (2 * inter + 1e-8) / (mask_pred_bin.sum() + mask_gt_bin.sum() + 1e-8)
        iou_v = (inter + 1e-8) / (union + 1e-8)

        detalles.append({
            "vertebra": vertebra_objetivo,
            "id_local": id_local,
            "id_real": VERTEBRA_TO_ID[vertebra_objetivo],
            "bbox_original": bbox_original,
            "bbox_expandida": bbox_expandida,
            "score_medsam": score_pred,
            "pix_gt": int(mask_gt_bin.sum()),
            "pix_pred": int(mask_pred_bin.sum()),
            "dice": float(dice_v),
            "iou": float(iou_v),
            "pred_vacia": bool(mask_pred_bin.sum() == 0),
            "gt_vacia": bool(mask_gt_bin.sum() == 0)
        })

    quality = {
        "overlap_pixels": int((pred_count > 1).sum()),
        "pred_pixels": int((pred_count > 0).sum()),
        "overlap_fraction_pred": float((pred_count > 1).sum() / max((pred_count > 0).sum(), 1)),
        "n_predicciones_vacias": int(sum(d["pred_vacia"] for d in detalles)),
        "n_gt_vacias": int(sum(d["gt_vacia"] for d in detalles))
    }

    if return_quality:
        return mask_sem_pred, mask_sem_gt, detalles, score_map, quality

    return mask_sem_pred, mask_sem_gt, detalles, score_map


def dice_multiclase_promedio(mask_pred, mask_gt, clases_ids):
    """Promedio macro: cada vertebra pesa igual, independiente de su area."""
    dices = []
    for cid in clases_ids:
        pred_bin = (mask_pred == cid).astype(np.uint8)
        gt_bin = (mask_gt == cid).astype(np.uint8)
        if gt_bin.sum() == 0 and pred_bin.sum() == 0:
            continue
        inter = np.logical_and(pred_bin == 1, gt_bin == 1).sum()
        dice = (2 * inter + 1e-8) / (pred_bin.sum() + gt_bin.sum() + 1e-8)
        dices.append(float(dice))
    return np.mean(dices) if len(dices) > 0 else np.nan


def iou_multiclase_promedio(mask_pred, mask_gt, clases_ids):
    ious = []
    for cid in clases_ids:
        pred_bin = (mask_pred == cid).astype(np.uint8)
        gt_bin = (mask_gt == cid).astype(np.uint8)
        if gt_bin.sum() == 0 and pred_bin.sum() == 0:
            continue
        inter = np.logical_and(pred_bin == 1, gt_bin == 1).sum()
        union = np.logical_or(pred_bin == 1, gt_bin == 1).sum()
        iou = (inter + 1e-8) / (union + 1e-8)
        ious.append(float(iou))
    return np.mean(ious) if len(ious) > 0 else np.nan


## Deteccion automatica de cajas

In [ ]:
# ============================================================
# 12.1) ConfiguraciÃ³n inicial y normalizaciÃ³n de prompts
# ============================================================

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

PROMPTS_BASE = PROMPTS_FILTRADOS if "PROMPTS_FILTRADOS" in globals() else PROMPTS

if "VERTEBRA_TO_ID" not in globals():
    VERTEBRA_TO_ID = {v: i + 1 for i, v in enumerate(CLASES_OBJETIVO)}

ID_TO_VERTEBRA = {v: k for k, v in VERTEBRA_TO_ID.items()}

if "N_CLASES" not in globals():
    N_CLASES = len(CLASES_OBJETIVO)


def normalizar_prompts_split(prompts_split):
    """
    Convierte los prompts a formato:
    {patient_id: prompts_sample}
    """

    if isinstance(prompts_split, dict):
        return prompts_split

    if isinstance(prompts_split, list):
        return {
            item["patient_id"]: item["prompts"]
            for item in prompts_split
        }

    raise TypeError("Formato de prompts no reconocido.")


PROMPTS_DICC = {
    split: normalizar_prompts_split(PROMPTS_BASE[split])
    for split in PROMPTS_BASE.keys()
}

print("Splits disponibles:", PROMPTS_DICC.keys())
print("Ejemplo val:", list(PROMPTS_DICC["val"].keys())[:3])
# ============================================================
# 12.2) Plantilla anatÃ³mica desde train
# ============================================================

def obtener_vertebra_desde_prompt(clave_prompt, info_prompt):
    """
    Obtiene el nombre de la vÃ©rtebra desde el prompt.
    """

    if "vertebra" in info_prompt:
        return info_prompt["vertebra"]

    try:
        clase_id = int(clave_prompt)
        return ID_TO_VERTEBRA[clase_id]
    except Exception:
        return clave_prompt


def construir_template_bbox_train(prompts_dicc, split_template="train"):
    """
    Construye una plantilla promedio de posiciÃ³n y tamaÃ±o por vÃ©rtebra.

    Se usa solo train. No usa val/test para construir la plantilla.
    """

    filas = []

    for patient_id, prompts_sample in prompts_dicc[split_template].items():

        for clave_prompt, info_prompt in prompts_sample.items():

            vertebra = obtener_vertebra_desde_prompt(clave_prompt, info_prompt)

            if vertebra not in CLASES_OBJETIVO:
                continue

            x0, y0, x1, y1 = info_prompt["bbox_xyxy"]

            filas.append({
                "patient_id": patient_id,
                "vertebra": vertebra,
                "id_real": VERTEBRA_TO_ID[vertebra],
                "cx_rel": ((x0 + x1) / 2) / 1024,
                "cy_rel": ((y0 + y1) / 2) / 1024,
                "w_rel": (x1 - x0) / 1024,
                "h_rel": (y1 - y0) / 1024
            })

    df_template = pd.DataFrame(filas)

    template_bbox = (
        df_template
        .groupby(["vertebra", "id_real"], as_index=False)
        .agg(
            cx_rel=("cx_rel", "median"),
            cy_rel=("cy_rel", "median"),
            w_rel=("w_rel", "median"),
            h_rel=("h_rel", "median")
        )
        .sort_values("id_real")
        .reset_index(drop=True)
    )

    faltantes = [v for v in CLASES_OBJETIVO if v not in template_bbox["vertebra"].tolist()]

    if len(faltantes) > 0:
        print("Advertencia: faltan vÃ©rtebras en la plantilla:", faltantes)

    return template_bbox, df_template


template_bbox_auto, df_template_bbox_train = construir_template_bbox_train(
    PROMPTS_DICC,
    split_template="train"
)

display(template_bbox_auto)

In [ ]:
# ============================================================
# 12.3) Utilidades para imagen, suavizado y ROI
# ============================================================

def imagen_a_gris_uint8(img_rgb):
    """
    Convierte imagen RGB o gris a uint8.
    """

    if img_rgb.ndim == 3:
        gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    else:
        gray = img_rgb.copy()

    if gray.max() <= 1.0:
        gray = gray * 255

    return gray.astype(np.uint8)


def normalizar_01(x, eps=1e-8):
    x = x.astype(np.float32)
    return (x - x.min()) / (x.max() - x.min() + eps)


def suavizar_1d(x, kernel_size=21):
    kernel_size = int(kernel_size)

    if kernel_size % 2 == 0:
        kernel_size += 1

    kernel = np.ones(kernel_size, dtype=np.float32) / kernel_size
    return np.convolve(x, kernel, mode="same")


def detectar_roi_radiografia(img_rgb, margen_x=35, margen_y=5):
    """
    Detecta el Ã¡rea no negra de la radiografÃ­a.
    Sirve para no buscar bordes en el fondo negro.
    """

    gray = imagen_a_gris_uint8(img_rgb)
    H, W = gray.shape

    valores = gray[gray > 0]

    if len(valores) == 0:
        return [0, 0, W - 1, H - 1]

    thr = max(5, np.percentile(valores, 3))
    mask = (gray > thr).astype(np.uint8)

    kernel = np.ones((21, 21), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask)

    if num_labels <= 1:
        return [0, 0, W - 1, H - 1]

    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])

    x, y, w, h, area = stats[largest]

    if w < W * 0.10 or h < H * 0.20:
        return [0, 0, W - 1, H - 1]

    x0 = max(0, x - margen_x)
    y0 = max(0, y - margen_y)
    x1 = min(W - 1, x + w + margen_x)
    y1 = min(H - 1, y + h + margen_y)

    return [x0, y0, x1, y1]


def seleccionar_picos_1d(profile, xs_abs, n_picos=8, min_sep=18):
    """
    Selecciona picos separados en un perfil 1D.
    """

    if len(profile) == 0:
        return np.array([]), np.array([])

    order = np.argsort(profile)[::-1]

    picos_x = []
    picos_s = []

    for idx in order:
        x = float(xs_abs[idx])
        s = float(profile[idx])

        if all(abs(x - px) >= min_sep for px in picos_x):
            picos_x.append(x)
            picos_s.append(s)

        if len(picos_x) >= n_picos:
            break

    return np.array(picos_x, dtype=np.float32), np.array(picos_s, dtype=np.float32)
# ============================================================
# 12.4) Estimar eje curvo desde bordes laterales
# ============================================================

def estimar_eje_curvo_por_bordes(
    img_rgb,
    template_bbox=None,
    n_franjas=27,
    search_half_frac=0.23,
    n_picos=8,
    min_sep_frac=0.018,
    poly_degree=3,
    debug=False
):
    """
    Estima un eje curvo de columna a partir de bordes laterales.

    LÃ³gica:
    1. Detecta ROI de radiografÃ­a.
    2. Divide la imagen en franjas horizontales.
    3. En cada franja busca candidatos de borde izquierdo y derecho.
    4. Selecciona el par de bordes mÃ¡s razonable.
    5. Calcula centro = (borde_izquierdo + borde_derecho) / 2.
    6. Suaviza los centros con un polinomio.
    """

    gray = imagen_a_gris_uint8(img_rgb)
    H, W = gray.shape

    x_roi0, y_roi0, x_roi1, y_roi1 = detectar_roi_radiografia(img_rgb)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray_eq = clahe.apply(gray)

    grad_x = np.abs(cv2.Sobel(gray_eq, cv2.CV_32F, 1, 0, ksize=3))
    grad_y = np.abs(cv2.Sobel(gray_eq, cv2.CV_32F, 0, 1, ksize=3))

    score_img = (
        0.70 * normalizar_01(grad_x) +
        0.20 * normalizar_01(gray_eq) +
        0.10 * normalizar_01(grad_y)
    )

    if template_bbox is not None:
        x_prior_inicial = float(np.median(template_bbox["cx_rel"].to_numpy(dtype=float) * W))
        ancho_esperado = float(np.median(template_bbox["w_rel"].to_numpy(dtype=float) * W))
    else:
        x_prior_inicial = (x_roi0 + x_roi1) / 2
        ancho_esperado = W * 0.10

    ancho_esperado = np.clip(ancho_esperado * 1.15, W * 0.06, W * 0.18)

    min_width = max(W * 0.045, ancho_esperado * 0.55)
    max_width = min(W * 0.28, ancho_esperado * 2.60)

    search_half = int(W * search_half_frac)
    min_sep = int(W * min_sep_frac)

    y_edges = np.linspace(0, H, n_franjas + 1).astype(int)

    puntos_centro = []
    puntos_izq = []
    puntos_der = []
    debug_rows = []

    x_prior = x_prior_inicial

    for i in range(n_franjas):

        y0 = int(y_edges[i])
        y1 = int(y_edges[i + 1])
        y_mid = int((y0 + y1) / 2)

        x0 = max(x_roi0, int(x_prior - search_half))
        x1 = min(x_roi1, int(x_prior + search_half))

        if x1 <= x0 + 10:
            x0 = max(0, int(x_prior - search_half))
            x1 = min(W - 1, int(x_prior + search_half))

        xs_abs = np.arange(x0, x1 + 1)

        crop_score = score_img[y0:y1, x0:x1 + 1]

        if crop_score.size == 0:
            continue

        profile = crop_score.mean(axis=0)
        profile = suavizar_1d(profile, kernel_size=max(15, int(W * 0.025)))
        profile = normalizar_01(profile)

        left_mask = xs_abs < (x_prior - min_width * 0.20)
        right_mask = xs_abs > (x_prior + min_width * 0.20)

        xs_left = xs_abs[left_mask]
        prof_left = profile[left_mask]

        xs_right = xs_abs[right_mask]
        prof_right = profile[right_mask]

        left_peaks_x, left_peaks_s = seleccionar_picos_1d(
            prof_left,
            xs_left,
            n_picos=n_picos,
            min_sep=min_sep
        )

        right_peaks_x, right_peaks_s = seleccionar_picos_1d(
            prof_right,
            xs_right,
            n_picos=n_picos,
            min_sep=min_sep
        )

        mejor = None
        mejor_score = -np.inf

        for xl, sl in zip(left_peaks_x, left_peaks_s):
            for xr, sr in zip(right_peaks_x, right_peaks_s):

                if xr <= xl:
                    continue

                width = xr - xl

                if width < min_width or width > max_width:
                    continue

                centro = (xl + xr) / 2

                penal_width = ((width - ancho_esperado) / (ancho_esperado + 1e-6)) ** 2
                penal_prior = ((centro - x_prior) / W) ** 2

                score = sl + sr - 0.75 * penal_width - 1.50 * penal_prior

                if score > mejor_score:
                    mejor_score = score
                    mejor = (xl, xr, centro, width, sl, sr)

        if mejor is None:
            xl = x_prior - ancho_esperado / 2
            xr = x_prior + ancho_esperado / 2
            centro = x_prior
            width = ancho_esperado
            sl = np.nan
            sr = np.nan
        else:
            xl, xr, centro, width, sl, sr = mejor

        puntos_izq.append([y_mid, xl])
        puntos_der.append([y_mid, xr])
        puntos_centro.append([y_mid, centro])

        debug_rows.append({
            "franja": i,
            "y_mid": y_mid,
            "x_left": xl,
            "x_right": xr,
            "x_center": centro,
            "width": width,
            "x_prior": x_prior,
            "score_left": sl,
            "score_right": sr,
            "score_pair": mejor_score
        })

        # ActualizaciÃ³n suave del prior para permitir escoliosis sin saltos bruscos.
        x_prior = 0.65 * x_prior + 0.35 * centro

    puntos_centro = np.array(puntos_centro, dtype=np.float32)
    puntos_izq = np.array(puntos_izq, dtype=np.float32)
    puntos_der = np.array(puntos_der, dtype=np.float32)

    if len(puntos_centro) < 4:
        raise ValueError("No se pudieron estimar suficientes puntos para el eje curvo.")

    y_raw = puntos_centro[:, 0]
    x_raw = puntos_centro[:, 1]

    deg = min(poly_degree, len(y_raw) - 1)
    coef = np.polyfit(y_raw, x_raw, deg=deg)
    polinomio = np.poly1d(coef)

    y_smooth = np.linspace(0, H - 1, 300)
    x_smooth = np.clip(polinomio(y_smooth), 0, W - 1)

    puntos_smooth = np.column_stack([y_smooth, x_smooth])

    info = {
        "puntos_centro_raw": puntos_centro,
        "puntos_izq": puntos_izq,
        "puntos_der": puntos_der,
        "puntos_smooth": puntos_smooth,
        "polinomio": polinomio,
        "roi": [x_roi0, y_roi0, x_roi1, y_roi1],
        "score_img": score_img,
        "debug": pd.DataFrame(debug_rows)
    }

    return info
# ============================================================
# 12.5) Generar prompts automÃ¡ticos T1-L5
# versiÃ³n corregida con calibraciÃ³n vertical no lineal
# ============================================================



# ============================================================
# 12.4B) Estimar eje curvo por respuesta central de columna
# ============================================================

def estimar_eje_curvo_por_respuesta_central(
    img_rgb,
    template_bbox=None,
    n_franjas=34,
    search_half_frac=0.20,
    continuidad_px=42,
    poly_degree=5,
    smooth_kernel=5,
    debug=False,
):
    gray = imagen_a_gris_uint8(img_rgb)
    H, W = gray.shape
    x_roi0, y_roi0, x_roi1, y_roi1 = detectar_roi_radiografia(img_rgb)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray_eq = clahe.apply(gray)
    grad_x = np.abs(cv2.Sobel(gray_eq, cv2.CV_32F, 1, 0, ksize=3))
    grad_y = np.abs(cv2.Sobel(gray_eq, cv2.CV_32F, 0, 1, ksize=3))
    lap = np.abs(cv2.Laplacian(gray_eq, cv2.CV_32F, ksize=3))

    score_img = (
        0.35 * normalizar_01(gray_eq) +
        0.30 * normalizar_01(grad_x) +
        0.20 * normalizar_01(grad_y) +
        0.15 * normalizar_01(lap)
    )

    if template_bbox is not None:
        x_prior = float(np.median(template_bbox["cx_rel"].to_numpy(dtype=float) * W))
        y_min = float(template_bbox["cy_rel"].min() * H)
        y_max = float(template_bbox["cy_rel"].max() * H)
    else:
        x_prior = (x_roi0 + x_roi1) / 2
        y_min, y_max = y_roi0, y_roi1

    y0_busq = max(0, int(min(y_roi0, y_min - 0.06 * H)))
    y1_busq = min(H - 1, int(max(y_roi1, y_max + 0.06 * H)))
    y_edges = np.linspace(y0_busq, y1_busq, n_franjas + 1).astype(int)
    search_half = int(W * search_half_frac)

    puntos = []
    debug_rows = []

    for i in range(n_franjas):
        y0 = int(y_edges[i])
        y1 = int(y_edges[i + 1])
        y_mid = int((y0 + y1) / 2)

        x0 = max(x_roi0, int(x_prior - search_half))
        x1 = min(x_roi1, int(x_prior + search_half))
        if x1 <= x0 + 10:
            x0 = max(0, int(x_prior - search_half))
            x1 = min(W - 1, int(x_prior + search_half))

        xs_abs = np.arange(x0, x1 + 1)
        crop = score_img[y0:y1, x0:x1 + 1]
        if crop.size == 0:
            continue

        profile = crop.mean(axis=0)
        profile = suavizar_1d(profile, kernel_size=max(9, int(W * 0.015)))
        profile = normalizar_01(profile)

        penal_continuidad = ((xs_abs - x_prior) / max(continuidad_px, 1)) ** 2
        score = profile - 0.18 * penal_continuidad
        best_idx = int(np.argmax(score))
        x_best = float(xs_abs[best_idx])

        puntos.append([y_mid, x_best])
        debug_rows.append({
            "franja": i,
            "y_mid": y_mid,
            "x_center": x_best,
            "x_prior": x_prior,
            "score_best": float(score[best_idx]),
            "profile_best": float(profile[best_idx]),
            "x0": x0,
            "x1": x1,
        })

        x_prior = 0.50 * x_prior + 0.50 * x_best

    puntos = np.array(puntos, dtype=np.float32)
    if len(puntos) < 5:
        raise ValueError("No se pudieron estimar suficientes puntos para el eje central.")

    y_raw = puntos[:, 0]
    x_raw = puntos[:, 1]
    x_smooth_raw = suavizar_1d(x_raw, kernel_size=smooth_kernel) if smooth_kernel and smooth_kernel > 1 else x_raw.copy()

    deg = min(poly_degree, len(y_raw) - 1)
    coef = np.polyfit(y_raw, x_smooth_raw, deg=deg)
    polinomio = np.poly1d(coef)

    y_smooth = np.linspace(0, H - 1, 300)
    x_smooth = np.clip(polinomio(y_smooth), 0, W - 1)
    puntos_smooth = np.column_stack([y_smooth, x_smooth])

    return {
        "puntos_centro_raw": puntos,
        "puntos_izq": puntos.copy(),
        "puntos_der": puntos.copy(),
        "puntos_smooth": puntos_smooth,
        "polinomio": polinomio,
        "roi": [x_roi0, y_roi0, x_roi1, y_roi1],
        "score_img": score_img,
        "debug": pd.DataFrame(debug_rows),
        "metodo_eje": "central",
    }




def _filtrar_puntos_eje_robusto(puntos, max_salto_px=55, ventana_mediana=5):
    """Filtra puntos centrales con saltos laterales extremos y suaviza localmente."""
    puntos = np.asarray(puntos, dtype=np.float32)
    if len(puntos) < 5:
        return puntos

    y = puntos[:, 0].copy()
    x = puntos[:, 1].copy()

    # Reemplazar saltos locales grandes por interpolacion de vecinos confiables.
    keep = np.ones(len(x), dtype=bool)
    for i in range(1, len(x)):
        if abs(x[i] - x[i - 1]) > max_salto_px:
            keep[i] = False

    if keep.sum() >= 4:
        x = np.interp(y, y[keep], x[keep])

    # Mediana movil: preserva curvas suaves y reduce puntos atraidos por costillas/pelvis.
    k = int(ventana_mediana)
    if k > 1:
        if k % 2 == 0:
            k += 1
        pad = k // 2
        x_pad = np.pad(x, (pad, pad), mode="edge")
        x_med = np.array([np.median(x_pad[i:i + k]) for i in range(len(x))])
        x = 0.65 * x_med + 0.35 * x

    return np.column_stack([y, x]).astype(np.float32)


def estimar_eje_curvo_por_respuesta_central_robusta(
    img_rgb,
    template_bbox=None,
    n_franjas=34,
    search_half_frac=0.22,
    continuidad_px=50,
    max_salto_px=55,
    ventana_mediana=5,
    debug=False,
):
    """
    Variante robusta del eje central.

    Usa los puntos por respuesta central, pero evita ajustar un polinomio global.
    La curva final es interpolacion local de puntos filtrados, por lo que no extrapola arcos raros en extremos.
    """
    info = estimar_eje_curvo_por_respuesta_central(
        img_rgb=img_rgb,
        template_bbox=template_bbox,
        n_franjas=n_franjas,
        search_half_frac=search_half_frac,
        continuidad_px=continuidad_px,
        poly_degree=3,
        smooth_kernel=1,
        debug=debug,
    )

    puntos_raw = info["puntos_centro_raw"]
    puntos_filtrados = _filtrar_puntos_eje_robusto(
        puntos_raw,
        max_salto_px=max_salto_px,
        ventana_mediana=ventana_mediana,
    )

    H, W = img_rgb.shape[:2]
    y_raw = puntos_filtrados[:, 0]
    x_raw = puntos_filtrados[:, 1]

    y_smooth = np.linspace(0, H - 1, 300)

    # Evitar extrapolacion agresiva: fuera del rango observado, mantener el extremo mas cercano.
    x_smooth = np.interp(y_smooth, y_raw, x_raw, left=x_raw[0], right=x_raw[-1])
    x_smooth = suavizar_1d(x_smooth, kernel_size=11)
    x_smooth = np.clip(x_smooth, 0, W - 1)

    # Polinomio compatible con el resto del codigo, construido sobre la curva ya robusta.
    coef = np.polyfit(y_smooth, x_smooth, deg=5)
    polinomio = np.poly1d(coef)

    info["puntos_centro_raw"] = puntos_filtrados
    info["puntos_izq"] = puntos_filtrados.copy()
    info["puntos_der"] = puntos_filtrados.copy()
    info["puntos_smooth"] = np.column_stack([y_smooth, x_smooth])
    info["polinomio"] = polinomio
    info["metodo_eje"] = "central_robusto"
    info["debug_raw"] = info.get("debug", pd.DataFrame()).copy()
    return info



def estimar_eje_por_ruta_dinamica(
    img_rgb,
    template_bbox=None,
    n_franjas=64,
    search_half_frac=0.32,
    n_candidatos=45,
    penal_salto=0.018,
    penal_curvatura=0.010,
    penal_prior=0.002,
    smooth_kernel=7,
    x_prior_override=None,
    ruta_nombre="ruta_dinamica",
    debug=False,
):
    """
    Eje por ruta dinamica: busca una trayectoria global de alta respuesta visual.

    A diferencia del metodo central greedy, no decide cada franja de forma aislada.
    Optimiza una ruta completa penalizando saltos y curvatura brusca.
    """
    gray = imagen_a_gris_uint8(img_rgb)
    H, W = gray.shape
    x_roi0, y_roi0, x_roi1, y_roi1 = detectar_roi_radiografia(img_rgb)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray_eq = clahe.apply(gray)
    grad_x = np.abs(cv2.Sobel(gray_eq, cv2.CV_32F, 1, 0, ksize=3))
    grad_y = np.abs(cv2.Sobel(gray_eq, cv2.CV_32F, 0, 1, ksize=3))
    lap = np.abs(cv2.Laplacian(gray_eq, cv2.CV_32F, ksize=3))

    score_img = (
        0.35 * normalizar_01(gray_eq) +
        0.30 * normalizar_01(grad_x) +
        0.20 * normalizar_01(grad_y) +
        0.15 * normalizar_01(lap)
    )

    if template_bbox is not None:
        x_template_prior = float(np.median(template_bbox["cx_rel"].to_numpy(dtype=float) * W))
        y_min = float(template_bbox["cy_rel"].min() * H)
        y_max = float(template_bbox["cy_rel"].max() * H)
    else:
        x_template_prior = (x_roi0 + x_roi1) / 2
        y_min, y_max = y_roi0, y_roi1

    if x_prior_override is not None:
        x_template_prior = float(np.clip(x_prior_override, x_roi0, x_roi1))

    y0_busq = max(0, int(min(y_roi0, y_min - 0.08 * H)))
    y1_busq = min(H - 1, int(max(y_roi1, y_max + 0.08 * H)))
    y_edges = np.linspace(y0_busq, y1_busq, n_franjas + 1).astype(int)

    search_half = int(W * search_half_frac)
    x0_global = max(x_roi0, int(x_template_prior - search_half))
    x1_global = min(x_roi1, int(x_template_prior + search_half))
    if x1_global <= x0_global + 20:
        x0_global, x1_global = x_roi0, x_roi1

    candidatas = []
    debug_rows = []

    for i in range(n_franjas):
        y0 = int(y_edges[i])
        y1 = int(y_edges[i + 1])
        y_mid = int((y0 + y1) / 2)
        xs_abs = np.arange(x0_global, x1_global + 1)
        crop = score_img[y0:y1, x0_global:x1_global + 1]
        if crop.size == 0:
            continue

        profile = crop.mean(axis=0)
        profile = suavizar_1d(profile, kernel_size=max(9, int(W * 0.012)))
        profile = normalizar_01(profile)

        # Mantener candidatos diversos: top por score y muestreo uniforme.
        top_idx = np.argsort(profile)[::-1][:n_candidatos]
        uniform_idx = np.linspace(0, len(xs_abs) - 1, min(n_candidatos // 2, len(xs_abs))).astype(int)
        idxs = np.unique(np.r_[top_idx, uniform_idx])
        xs = xs_abs[idxs].astype(float)
        scores = profile[idxs].astype(float)

        order = np.argsort(xs)
        xs = xs[order]
        scores = scores[order]

        candidatas.append({"y": float(y_mid), "xs": xs, "scores": scores})
        debug_rows.append({"franja": i, "y_mid": y_mid, "n_candidatos": len(xs), "score_max": float(scores.max())})

    if len(candidatas) < 5:
        raise ValueError("No hay suficientes franjas candidatas para ruta dinamica.")

    # DP de segundo orden aproximado: estado = candidato actual, guarda mejor predecesor.
    dp = []
    back = []

    xs0 = candidatas[0]["xs"]
    sc0 = candidatas[0]["scores"]
    prior0 = -penal_prior * ((xs0 - x_template_prior) / W) ** 2
    dp.append(sc0 + prior0)
    back.append(np.full(len(xs0), -1, dtype=int))

    prev_prev_xs = None
    prev_xs = xs0

    for t in range(1, len(candidatas)):
        xs = candidatas[t]["xs"]
        sc = candidatas[t]["scores"]
        prev_score = dp[-1]
        mat_salto = ((xs[:, None] - prev_xs[None, :]) / W) ** 2
        score_mat = prev_score[None, :] - penal_salto * mat_salto

        if t >= 2 and prev_prev_xs is not None:
            # Curvatura aproximada usando el mejor predecesor de cada prev.
            prev_back = back[-1]
            prevprev_for_prev = np.array([
                prev_prev_xs[j] if j >= 0 else prev_xs[k]
                for k, j in enumerate(prev_back)
            ])
            curv = (xs[:, None] - 2 * prev_xs[None, :] + prevprev_for_prev[None, :])
            score_mat -= penal_curvatura * (curv / W) ** 2

        best_prev = np.argmax(score_mat, axis=1)
        best_score = score_mat[np.arange(len(xs)), best_prev] + sc
        best_score -= penal_prior * ((xs - x_template_prior) / W) ** 2

        dp.append(best_score)
        back.append(best_prev.astype(int))
        prev_prev_xs = prev_xs
        prev_xs = xs

    # Backtracking.
    idx_last = int(np.argmax(dp[-1]))
    ruta_idx = [idx_last]
    for t in range(len(candidatas) - 1, 0, -1):
        idx_last = int(back[t][idx_last])
        ruta_idx.append(idx_last)
    ruta_idx = ruta_idx[::-1]

    puntos = []
    for item, idx_cand in zip(candidatas, ruta_idx):
        puntos.append([item["y"], item["xs"][idx_cand]])
    puntos = np.array(puntos, dtype=np.float32)

    puntos_filtrados = _filtrar_puntos_eje_robusto(puntos, max_salto_px=65, ventana_mediana=smooth_kernel)
    y_raw = puntos_filtrados[:, 0]
    x_raw = puntos_filtrados[:, 1]

    y_smooth = np.linspace(0, H - 1, 300)
    x_smooth = np.interp(y_smooth, y_raw, x_raw, left=x_raw[0], right=x_raw[-1])
    x_smooth = suavizar_1d(x_smooth, kernel_size=13)
    x_smooth = np.clip(x_smooth, 0, W - 1)

    coef = np.polyfit(y_smooth, x_smooth, deg=5)
    polinomio = np.poly1d(coef)

    return {
        "puntos_centro_raw": puntos_filtrados,
        "puntos_izq": puntos_filtrados.copy(),
        "puntos_der": puntos_filtrados.copy(),
        "puntos_smooth": np.column_stack([y_smooth, x_smooth]),
        "polinomio": polinomio,
        "roi": [x_roi0, y_roi0, x_roi1, y_roi1],
        "score_img": score_img,
        "debug": pd.DataFrame(debug_rows),
        "x_prior_usado": float(x_template_prior),
        "metodo_eje": ruta_nombre,
    }

def estimar_eje_columna(img_rgb, template_bbox=None, metodo="bordes"):
    if metodo == "bordes":
        info = estimar_eje_curvo_por_bordes(img_rgb, template_bbox=template_bbox)
        info["metodo_eje"] = "bordes"
        return info
    if metodo == "central":
        return estimar_eje_curvo_por_respuesta_central(img_rgb, template_bbox=template_bbox)
    if metodo == "central_robusto":
        return estimar_eje_curvo_por_respuesta_central_robusta(img_rgb, template_bbox=template_bbox)
    if metodo == "ruta_dinamica":
        return estimar_eje_por_ruta_dinamica(img_rgb, template_bbox=template_bbox)
    raise ValueError(f"metodo_eje no reconocido: {metodo}")




def limitar_info_eje_a_template(info_eje, img_shape, template_bbox=None, margen_rel=0.05):
    """Recorta puntos/curva del eje al rango vertical esperado T1-L5 para evitar craneo/cadera."""
    if template_bbox is None:
        return info_eje

    H = img_shape[0]
    y_min = float(template_bbox["cy_rel"].min() * H)
    y_max = float(template_bbox["cy_rel"].max() * H)
    margen = float(margen_rel * H)
    y0 = max(0, y_min - margen)
    y1 = min(H - 1, y_max + margen)

    info = dict(info_eje)
    for key in ["puntos_centro_raw", "puntos_izq", "puntos_der", "puntos_smooth"]:
        if key in info and info[key] is not None:
            pts = np.asarray(info[key])
            if pts.ndim == 2 and pts.shape[1] >= 2:
                keep = (pts[:, 0] >= y0) & (pts[:, 0] <= y1)
                if keep.sum() >= 2:
                    info[key] = pts[keep]

    # Reconstruir interpolador solo dentro del rango anatomico si hay puntos suficientes.
    pts_curve = np.asarray(info.get("puntos_smooth", []))
    if pts_curve.ndim == 2 and len(pts_curve) >= 4:
        y = pts_curve[:, 0]
        x = pts_curve[:, 1]
        def curva_limitada(yq, y=y, x=x):
            return np.interp(yq, y, x, left=x[0], right=x[-1])
        info["polinomio"] = curva_limitada

    info["rango_y_anatomico"] = [float(y0), float(y1)]
    return info

def generar_prompts_auto_por_bordes(
    img_rgb,
    template_bbox,
    escala_w=1.90,
    escala_h=1.35,
    peso_eje=0.90,
    escala_pos_y=0.91,
    offset_y=-16,
    offset_x=-18,
    mid_lift=22,
    min_w_frac=0.10,
    min_h_frac=0.035,
    max_w_frac=0.34,
    max_h_frac=0.13,
    metodo_eje="bordes",
    limitar_rango_y_anatomico=True,
    debug=False
):
    """
    Genera cajas automÃ¡ticas para las 17 vÃ©rtebras.

    Correcciones incluidas:
    - El eje horizontal se estima desde bordes laterales.
    - Las claves son 'T1', 'T2', ..., 'L5'.
    - Se aplica una compresiÃ³n vertical de la plantilla.
    - Se aplica una correcciÃ³n vertical no lineal para subir mÃ¡s la zona media.
    - Se aplica un pequeÃ±o desplazamiento horizontal hacia la izquierda.
    """

    H, W = img_rgb.shape[:2]

    info_eje = estimar_eje_columna(
        img_rgb,
        template_bbox=template_bbox,
        metodo=metodo_eje
    )

    if limitar_rango_y_anatomico:
        info_eje = limitar_info_eje_a_template(
            info_eje,
            img_rgb.shape,
            template_bbox=template_bbox,
            margen_rel=0.05,
        )

    curva = info_eje["polinomio"]

    prompts_auto = {}
    filas_debug = []

    template_ordenado = template_bbox.sort_values("id_real").reset_index(drop=True)

    y_anchor = float(template_ordenado.iloc[0]["cy_rel"] * H)
    y_min_template = float(template_ordenado["cy_rel"].min() * H)
    y_max_template = float(template_ordenado["cy_rel"].max() * H)

    for _, row in template_ordenado.iterrows():

        vertebra = row["vertebra"]

        if vertebra not in CLASES_OBJETIVO:
            continue

        id_real = int(row["id_real"])

        # PosiciÃ³n vertical original de la plantilla.
        cy_original = float(row["cy_rel"] * H)

        # PosiciÃ³n relativa entre T1 y L5.
        t = (cy_original - y_min_template) / (y_max_template - y_min_template + 1e-6)
        t = float(np.clip(t, 0, 1))

        # CorrecciÃ³n no lineal:
        # levanta mÃ¡s la zona media y casi no modifica los extremos.
        correccion_media = mid_lift * np.sin(np.pi * t)

        # CorrecciÃ³n vertical final.
        cy = y_anchor + escala_pos_y * (cy_original - y_anchor) + offset_y - correccion_media
        cy = float(np.clip(cy, 0, H - 1))

        # Centro horizontal desde el eje curvo, evaluado en la altura corregida.
        cx_eje = float(curva(cy))

        # Centro horizontal promedio desde plantilla.
        cx_template = float(row["cx_rel"] * W)

        # Mezcla eje automÃ¡tico + plantilla, con correcciÃ³n horizontal.
        cx = peso_eje * cx_eje + (1 - peso_eje) * cx_template + offset_x
        cx = float(np.clip(cx, 0, W - 1))

        # TamaÃ±o de caja.
        bw = float(row["w_rel"] * W * escala_w)
        bh = float(row["h_rel"] * H * escala_h)

        bw = float(np.clip(bw, W * min_w_frac, W * max_w_frac))
        bh = float(np.clip(bh, H * min_h_frac, H * max_h_frac))

        x0 = int(round(cx - bw / 2))
        x1 = int(round(cx + bw / 2))
        y0 = int(round(cy - bh / 2))
        y1 = int(round(cy + bh / 2))

        x0 = max(0, x0)
        y0 = max(0, y0)
        x1 = min(W - 1, x1)
        y1 = min(H - 1, y1)

        prompts_auto[vertebra] = {
            "vertebra": vertebra,
            "id_real": id_real,
            "bbox_xyxy": [x0, y0, x1, y1],
            "prompt_origen": "bbox_auto_bordes_eje_curvo_y_calibrado"
        }

        filas_debug.append({
            "vertebra": vertebra,
            "id_real": id_real,
            "cy_original": cy_original,
            "cy_final": cy,
            "t_vertical": t,
            "correccion_media": correccion_media,
            "cx_eje": cx_eje,
            "cx_template": cx_template,
            "cx_final": cx,
            "escala_pos_y": escala_pos_y,
            "offset_y": offset_y,
            "offset_x": offset_x,
            "mid_lift": mid_lift,
            "bbox_xyxy": [x0, y0, x1, y1]
        })

    df_debug = pd.DataFrame(filas_debug)

    if debug:
        return prompts_auto, info_eje, df_debug

    return prompts_auto, info_eje

In [ ]:
# ============================================================
# 12.6) VisualizaciÃ³n de eje, bordes y cajas automÃ¡ticas
# ============================================================

def visualizar_cajas_auto_bordes(
    img_rgb,
    prompts_auto,
    info_eje=None,
    mascara_gt=None,
    titulo="Cajas automÃ¡ticas por bordes laterales"
):
    plt.figure(figsize=(8, 10))
    plt.imshow(img_rgb)

    ax = plt.gca()

    if mascara_gt is not None:
        gt_overlay = np.zeros_like(img_rgb)
        gt_overlay[..., 0] = (mascara_gt > 0).astype(np.uint8) * 255
        plt.imshow(gt_overlay, alpha=0.20)

    if info_eje is not None:
        x0, y0, x1, y1 = info_eje["roi"]

        rect_roi = plt.Rectangle(
            (x0, y0),
            x1 - x0,
            y1 - y0,
            fill=False,
            edgecolor="white",
            linewidth=1.2,
            linestyle="--"
        )
        ax.add_patch(rect_roi)

        metodo_eje = info_eje.get("metodo_eje", "bordes")
        puntos_centro = info_eje["puntos_centro_raw"]
        puntos_smooth = info_eje["puntos_smooth"]

        if metodo_eje == "bordes":
            puntos_izq = info_eje["puntos_izq"]
            puntos_der = info_eje["puntos_der"]
            plt.scatter(puntos_izq[:, 1], puntos_izq[:, 0], s=10, c="orange", label="Borde izquierdo")
            plt.scatter(puntos_der[:, 1], puntos_der[:, 0], s=10, c="yellow", label="Borde derecho")
            label_centro = "Centro por bordes"
        else:
            label_centro = "Centro por respuesta"

        plt.scatter(puntos_centro[:, 1], puntos_centro[:, 0], s=14, c="cyan", label=label_centro)
        plt.plot(puntos_smooth[:, 1], puntos_smooth[:, 0], c="cyan", linewidth=2, label=f"Eje {metodo_eje} suavizado")

    for vertebra, info in prompts_auto.items():

        x0, y0, x1, y1 = info["bbox_xyxy"]

        rect = plt.Rectangle(
            (x0, y0),
            x1 - x0,
            y1 - y0,
            fill=False,
            edgecolor="lime",
            linewidth=1.2
        )
        ax.add_patch(rect)

        ax.text(
            x0,
            max(0, y0 - 3),
            vertebra,
            fontsize=8,
            color="white",
            bbox=dict(facecolor="black", alpha=0.45, pad=1)
        )

    plt.title(titulo)
    plt.axis("off")
    plt.legend(loc="lower right")
    plt.show()
# ============================================================
# 12.7) Cobertura de cajas automÃ¡ticas
# ============================================================

def evaluar_cobertura_cajas_por_clase(prompts_auto, mascara_gt_multiclase):
    """
    Evalua la calidad espacial de cada caja automatica contra la mascara real.

    Metricas principales:
    - bbox_recall: cuanto de la vertebra real queda dentro de la caja.
    - bbox_precision: que proporcion del area de la caja corresponde a la vertebra.
    - bbox_iou: interseccion / union entre caja y mascara real.
    - bbox_area_ratio: tamano de caja relativo al area real; valores muy altos suelen indicar cajas demasiado grandes.
    """

    filas = []

    for vertebra in CLASES_OBJETIVO:

        if vertebra not in prompts_auto:
            continue

        info = prompts_auto[vertebra]
        x0, y0, x1, y1 = info["bbox_xyxy"]

        gt_i = construir_mask_binaria_vertebra(
            mascara_gt_multiclase,
            vertebra
        ).astype(bool)

        total_gt = int(gt_i.sum())

        cover = np.zeros_like(gt_i, dtype=bool)
        cover[y0:y1 + 1, x0:x1 + 1] = True

        bbox_area = int(cover.sum())
        inter = int(np.logical_and(gt_i, cover).sum())
        union = int(np.logical_or(gt_i, cover).sum())

        bbox_recall = inter / total_gt if total_gt > 0 else np.nan
        bbox_precision = inter / bbox_area if bbox_area > 0 else np.nan
        bbox_iou = inter / union if union > 0 else np.nan
        bbox_area_ratio = bbox_area / total_gt if total_gt > 0 else np.nan

        filas.append({
            "vertebra": vertebra,
            "id_real": VERTEBRA_TO_ID[vertebra],
            "bbox_auto": [x0, y0, x1, y1],
            "gt_area": total_gt,
            "bbox_area": bbox_area,
            "bbox_intersection": inter,
            "bbox_union": union,
            "bbox_recall": bbox_recall,
            "bbox_precision": bbox_precision,
            "bbox_iou": bbox_iou,
            "bbox_area_ratio": bbox_area_ratio,
        })

    return pd.DataFrame(filas).sort_values("id_real").reset_index(drop=True)

def diagnosticar_desplazamiento_cajas(prompts_auto, mascara_gt_multiclase):
    """
    DiagnÃ³stico: compara centro de caja vs centro de la mÃ¡scara real.
    Solo evaluaciÃ³n, no generaciÃ³n de cajas.
    """

    filas = []

    for vertebra in CLASES_OBJETIVO:

        if vertebra not in prompts_auto:
            continue

        x0, y0, x1, y1 = prompts_auto[vertebra]["bbox_xyxy"]

        cx_box = (x0 + x1) / 2
        cy_box = (y0 + y1) / 2

        gt_i = construir_mask_binaria_vertebra(
            mascara_gt_multiclase,
            vertebra
        ).astype(bool)

        if gt_i.sum() == 0:
            continue

        ys, xs = np.where(gt_i)

        cx_gt = xs.mean()
        cy_gt = ys.mean()

        filas.append({
            "vertebra": vertebra,
            "cx_box": cx_box,
            "cy_box": cy_box,
            "cx_gt": cx_gt,
            "cy_gt": cy_gt,
            "dx_box_gt": cx_box - cx_gt,
            "dy_box_gt": cy_box - cy_gt
        })

    return pd.DataFrame(filas)


# ============================================================
# 12.8) Refinamiento local de centros vertebrales
# ============================================================

def construir_score_local_vertebra(img_rgb):
    """Score visual para buscar centros oseos locales sin usar mascara GT."""
    gray = imagen_a_gris_uint8(img_rgb)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray_eq = clahe.apply(gray)
    grad_x = np.abs(cv2.Sobel(gray_eq, cv2.CV_32F, 1, 0, ksize=3))
    grad_y = np.abs(cv2.Sobel(gray_eq, cv2.CV_32F, 0, 1, ksize=3))
    lap = np.abs(cv2.Laplacian(gray_eq, cv2.CV_32F, ksize=3))
    score = (
        0.40 * normalizar_01(gray_eq) +
        0.25 * normalizar_01(grad_x) +
        0.20 * normalizar_01(grad_y) +
        0.15 * normalizar_01(lap)
    )
    return normalizar_01(score)


def refinar_prompts_por_respuesta_local(
    img_rgb,
    prompts_auto,
    max_dx=35,
    max_dy=24,
    ventana_factor=1.45,
    top_percentile=82,
    penal_distancia=0.55,
    conservar_tamano=True,
    debug=False,
):
    """
    Refina el centro de cada caja buscando respuesta local de vertebra cerca de la caja inicial.

    No usa mascara GT. La mascara se sigue usando solo para evaluar despues.
    """
    H, W = img_rgb.shape[:2]
    score_img = construir_score_local_vertebra(img_rgb)

    refinados = {}
    filas = []

    for vertebra, info in prompts_auto.items():
        x0, y0, x1, y1 = [int(v) for v in info["bbox_xyxy"]]
        bw = max(x1 - x0 + 1, 1)
        bh = max(y1 - y0 + 1, 1)
        cx0 = (x0 + x1) / 2
        cy0 = (y0 + y1) / 2

        wx = int(round(bw * ventana_factor / 2))
        wy = int(round(bh * ventana_factor / 2))
        vx0 = max(0, int(round(cx0 - wx)))
        vx1 = min(W - 1, int(round(cx0 + wx)))
        vy0 = max(0, int(round(cy0 - wy)))
        vy1 = min(H - 1, int(round(cy0 + wy)))

        crop = score_img[vy0:vy1 + 1, vx0:vx1 + 1].copy()
        if crop.size == 0:
            refinados[vertebra] = dict(info)
            continue

        yy, xx = np.mgrid[vy0:vy1 + 1, vx0:vx1 + 1]
        dx = (xx - cx0) / max(max_dx, 1)
        dy = (yy - cy0) / max(max_dy, 1)
        dist2 = dx ** 2 + dy ** 2

        # Limitar busqueda a una elipse alrededor de la caja inicial.
        mask_busqueda = (np.abs(xx - cx0) <= max_dx) & (np.abs(yy - cy0) <= max_dy)
        score = crop - penal_distancia * dist2
        score[~mask_busqueda] = -np.inf

        valid = np.isfinite(score)
        if not valid.any():
            refinados[vertebra] = dict(info)
            continue

        thr = np.percentile(score[valid], top_percentile)
        pesos = np.clip(score - thr, 0, None)
        if pesos.sum() <= 1e-8:
            best = np.unravel_index(np.nanargmax(score), score.shape)
            cy_ref = float(vy0 + best[0])
            cx_ref = float(vx0 + best[1])
        else:
            cx_ref = float((xx * pesos).sum() / pesos.sum())
            cy_ref = float((yy * pesos).sum() / pesos.sum())

        cx_ref = float(np.clip(cx_ref, cx0 - max_dx, cx0 + max_dx))
        cy_ref = float(np.clip(cy_ref, cy0 - max_dy, cy0 + max_dy))

        if conservar_tamano:
            bw_new, bh_new = bw, bh
        else:
            bw_new, bh_new = bw * 0.95, bh * 0.95

        nx0 = int(round(cx_ref - bw_new / 2))
        nx1 = int(round(cx_ref + bw_new / 2))
        ny0 = int(round(cy_ref - bh_new / 2))
        ny1 = int(round(cy_ref + bh_new / 2))

        nx0 = max(0, nx0); ny0 = max(0, ny0)
        nx1 = min(W - 1, nx1); ny1 = min(H - 1, ny1)

        nuevo = dict(info)
        nuevo["bbox_xyxy"] = [nx0, ny0, nx1, ny1]
        nuevo["bbox_xyxy_base"] = [x0, y0, x1, y1]
        nuevo["prompt_origen"] = str(info.get("prompt_origen", "bbox_auto")) + "_refinado_local"
        refinados[vertebra] = nuevo

        filas.append({
            "vertebra": vertebra,
            "cx_base": cx0,
            "cy_base": cy0,
            "cx_refinado": cx_ref,
            "cy_refinado": cy_ref,
            "dx_refinado": cx_ref - cx0,
            "dy_refinado": cy_ref - cy0,
            "bbox_base": [x0, y0, x1, y1],
            "bbox_refinada": [nx0, ny0, nx1, ny1],
        })

    df_debug = pd.DataFrame(filas)
    if debug:
        return refinados, df_debug, score_img
    return refinados


def comparar_refinamiento_local_muestra(
    split="val",
    patient_id="N_12",
    metodo_eje="central_robusto",
    params=None,
):
    if params is None:
        params = {
            "escala_w": 1.40,
            "escala_h": 1.20,
            "peso_eje": 0.90,
            "escala_pos_y": 0.91,
            "offset_y": -16,
            "offset_x": -10,
            "mid_lift": 15,
            "metodo_eje": metodo_eje,
        }

    path_img, path_mask = resolver_paths_muestra(split, patient_id)
    img = cargar_imagen(path_img)
    mask = cargar_mascara(path_mask)

    prompts_base, info_eje = generar_prompts_auto_por_bordes(
        img_rgb=img,
        template_bbox=template_bbox_auto,
        debug=False,
        **params,
    )
    prompts_ref, df_ref, score_img = refinar_prompts_por_respuesta_local(
        img,
        prompts_base,
        debug=True,
    )

    df_base = evaluar_cobertura_cajas_por_clase(prompts_base, mask)
    df_refinada = evaluar_cobertura_cajas_por_clase(prompts_ref, mask)

    resumen = pd.DataFrame([
        {
            "version": "base",
            "bbox_iou": df_base["bbox_iou"].mean(),
            "bbox_recall": df_base["bbox_recall"].mean(),
            "bbox_precision": df_base["bbox_precision"].mean(),
        },
        {
            "version": "refinada_local",
            "bbox_iou": df_refinada["bbox_iou"].mean(),
            "bbox_recall": df_refinada["bbox_recall"].mean(),
            "bbox_precision": df_refinada["bbox_precision"].mean(),
        }
    ])

    visualizar_cajas_auto_bordes(
        img_rgb=img,
        prompts_auto=prompts_base,
        info_eje=info_eje,
        mascara_gt=mask,
        titulo=f"{patient_id} - cajas base"
    )
    visualizar_cajas_auto_bordes(
        img_rgb=img,
        prompts_auto=prompts_ref,
        info_eje=info_eje,
        mascara_gt=mask,
        titulo=f"{patient_id} - cajas refinadas localmente"
    )

    return resumen, df_base, df_refinada, df_ref, prompts_base, prompts_ref


## Estrategia final

In [ ]:
# ============================================================
# Estrategia final compacta de deteccion
# ============================================================

EJECUTAR_PRUEBA_FINAL_CORTA = True
EJECUTAR_VISUAL_FINAL = False  # Activar despues; evita recalcular casos visuales durante la corrida principal.
EJECUTAR_MEDSAM_BASICO = False

PREFIJO_FINAL = "vertebra_semana_final_val_v3_ampliada"
PACIENTES_VISUALES_FINAL = ["N_12", "S_187", "S_130", "S_80"]


def tipo_desde_patient_id(patient_id):
    return "escoliosis" if str(patient_id).startswith("S_") else "normal"


EJES_FINAL_NORMAL = [
    {"nombre": "bordes", "tipo": "metodo", "metodo": "bordes"},
    {"nombre": "central_robusto", "tipo": "metodo", "metodo": "central_robusto"},
]

EJES_FINAL_ESCOLIOSIS = [
    {"nombre": "ruta_dinamica", "tipo": "metodo", "metodo": "ruta_dinamica"},
    {"nombre": "multi_-0.10", "tipo": "multi", "offset_frac": -0.10},
    {"nombre": "multi_-0.05", "tipo": "multi", "offset_frac": -0.05},
    {"nombre": "multi_+0.00", "tipo": "multi", "offset_frac": 0.00},
    {"nombre": "multi_+0.05", "tipo": "multi", "offset_frac": 0.05},
    {"nombre": "multi_+0.10", "tipo": "multi", "offset_frac": 0.10},
    {"nombre": "central_robusto", "tipo": "metodo", "metodo": "central_robusto"},
]


def estimar_info_eje_final(img_rgb, eje_cfg, search_half_frac=0.16):
    H, W = img_rgb.shape[:2]

    if eje_cfg["tipo"] == "metodo":
        return estimar_eje_columna(
            img_rgb,
            template_bbox=template_bbox_auto,
            metodo=eje_cfg["metodo"],
        )

    if eje_cfg["tipo"] == "multi":
        x_prior_base = float(np.median(template_bbox_auto["cx_rel"].to_numpy(dtype=float) * W))
        x_prior = x_prior_base + float(eje_cfg["offset_frac"]) * W
        return estimar_eje_por_ruta_dinamica(
            img_rgb,
            template_bbox=template_bbox_auto,
            search_half_frac=search_half_frac,
            x_prior_override=x_prior,
            ruta_nombre=eje_cfg["nombre"],
        )

    raise ValueError(f"Tipo de eje no reconocido: {eje_cfg}")


def generar_prompts_final_desde_info_eje(
    img_rgb,
    template_bbox,
    info_eje,
    escala_w=1.35,
    escala_h=1.15,
    peso_eje=1.00,
    escala_pos_y=1.00,
    offset_y=-20,
    offset_x=-10,
    mid_lift=24,
    global_lift=0,
    thoracic_lift=0,
    mid_spine_lift=0,
    lumbar_lift=0,
    min_w_frac=0.10,
    min_h_frac=0.035,
    max_w_frac=0.30,
    max_h_frac=0.12,
):
    """
    Convierte una curva de columna en cajas vertebrales.

    La correccion por tramos evita asumir que solo L5 esta corrida:
    - global_lift sube todas las vertebras.
    - thoracic_lift sube principalmente T1-T6.
    - mid_spine_lift sube principalmente T7-L1.
    - lumbar_lift sube principalmente L2-L5.
    """

    H, W = img_rgb.shape[:2]
    info_eje = limitar_info_eje_a_template(
        info_eje,
        img_rgb.shape,
        template_bbox=template_bbox,
        margen_rel=0.05,
    )

    curva = info_eje["polinomio"]
    prompts_auto = {}
    filas_debug = []
    template_ordenado = template_bbox.sort_values("id_real").reset_index(drop=True)

    y_anchor = float(template_ordenado.iloc[0]["cy_rel"] * H)
    y_min_template = float(template_ordenado["cy_rel"].min() * H)
    y_max_template = float(template_ordenado["cy_rel"].max() * H)

    for _, row in template_ordenado.iterrows():
        vertebra = row["vertebra"]
        if vertebra not in CLASES_OBJETIVO:
            continue

        id_real = int(row["id_real"])
        cy_original = float(row["cy_rel"] * H)
        t = (cy_original - y_min_template) / (y_max_template - y_min_template + 1e-6)
        t = float(np.clip(t, 0, 1))

        correccion_media = mid_lift * np.sin(np.pi * t)

        thoracic_w = float(np.clip((0.42 - t) / 0.42, 0, 1)) ** 1.2
        mid_w = float(np.clip(1.0 - abs(t - 0.58) / 0.30, 0, 1)) ** 1.2
        lumbar_w = float(np.clip((t - 0.62) / 0.38, 0, 1)) ** 1.3

        correccion_tramos = (
            global_lift +
            thoracic_lift * thoracic_w +
            mid_spine_lift * mid_w +
            lumbar_lift * lumbar_w
        )

        cy = y_anchor + escala_pos_y * (cy_original - y_anchor) + offset_y - correccion_media - correccion_tramos
        cy = float(np.clip(cy, 0, H - 1))

        cx_eje = float(curva(cy))
        cx_template = float(row["cx_rel"] * W)
        cx = peso_eje * cx_eje + (1 - peso_eje) * cx_template + offset_x
        cx = float(np.clip(cx, 0, W - 1))

        bw = float(row["w_rel"] * W * escala_w)
        bh = float(row["h_rel"] * H * escala_h)
        bw = float(np.clip(bw, W * min_w_frac, W * max_w_frac))
        bh = float(np.clip(bh, H * min_h_frac, H * max_h_frac))

        x0 = int(round(cx - bw / 2))
        x1 = int(round(cx + bw / 2))
        y0 = int(round(cy - bh / 2))
        y1 = int(round(cy + bh / 2))

        x0 = max(0, x0)
        y0 = max(0, y0)
        x1 = min(W - 1, x1)
        y1 = min(H - 1, y1)

        prompts_auto[vertebra] = {
            "vertebra": vertebra,
            "id_real": id_real,
            "bbox_xyxy": [x0, y0, x1, y1],
            "prompt_origen": "vertebra_semana_final",
        }

        filas_debug.append({
            "vertebra": vertebra,
            "id_real": id_real,
            "t_vertical": t,
            "cy_final": cy,
            "cx_final": cx,
            "cx_eje": cx_eje,
            "cx_template": cx_template,
            "correccion_media": correccion_media,
            "correccion_tramos": correccion_tramos,
            "bbox_xyxy": [x0, y0, x1, y1],
        })

    return prompts_auto, info_eje, pd.DataFrame(filas_debug)


def construir_grid_final(tipo_real, modo_rapido=True):
    """
    Busqueda ampliada v3.
    Recupera combinaciones que la corrida larga mostro utiles, sin volver a una grilla enorme.
    """
    filas = []

    if tipo_real == "normal":
        # Normales fueron estables; reabrimos lo que perdio N_32/N_9 sin hacerlo pesado.
        offsets = (-47, -43, -35, -27, -23, -15, -7, 5)
        escalas_y = (0.92, 0.96, 0.98, 1.00, 1.02)
        mid_vals = (0, 24, 36)
        lift_sets = [
            {"global_lift": 0, "thoracic_lift": 0, "mid_spine_lift": 0, "lumbar_lift": 0},
            {"global_lift": 8, "thoracic_lift": 0, "mid_spine_lift": 0, "lumbar_lift": 0},
        ]
        escalas_wh = ((1.35, 1.15),)
    else:
        # Escoliosis: recuperar offsets/escala_w que salvaron casos en la corrida larga.
        offsets = (-194, -182, -170, -157, -145, -133, -120, -107, -95, -69, -45, -33, -20, -8, 4)
        escalas_y = (0.84, 0.96, 0.98, 1.04)
        mid_vals = (0, 24, 36, 48)
        lift_sets = [
            # Baseline: en promedio fue el mas fuerte.
            {"global_lift": 0, "thoracic_lift": 0, "mid_spine_lift": 0, "lumbar_lift": 0},
            # Lifts discretos solo como hipotesis puntual, no como dogma.
            {"global_lift": 16, "thoracic_lift": 0, "mid_spine_lift": 0, "lumbar_lift": 0},
            {"global_lift": 0, "thoracic_lift": 0, "mid_spine_lift": 16, "lumbar_lift": 0},
            {"global_lift": 16, "thoracic_lift": 0, "mid_spine_lift": 0, "lumbar_lift": 24},
        ]
        escalas_wh = ((1.35, 1.15), (1.55, 1.25), (1.75, 1.35))

    for offset_y in offsets:
        for escala_pos_y in escalas_y:
            for mid_lift in mid_vals:
                for lifts in lift_sets:
                    for escala_w, escala_h in escalas_wh:
                        filas.append({
                            "escala_w": escala_w,
                            "escala_h": escala_h,
                            "peso_eje": 1.00,
                            "escala_pos_y": escala_pos_y,
                            "offset_y": offset_y,
                            "offset_x": -10,
                            "mid_lift": mid_lift,
                            **lifts,
                        })
    return filas

def ejes_final_por_tipo(tipo_real):
    return EJES_FINAL_ESCOLIOSIS if tipo_real == "escoliosis" else EJES_FINAL_NORMAL


def evaluar_estrategia_final_muestra(split, patient_id, modo_rapido=True):
    tipo_real = tipo_desde_patient_id(patient_id)
    path_img, path_mask = resolver_paths_muestra(split, patient_id)
    imagen = cargar_imagen(path_img)
    mascara = cargar_mascara(path_mask)

    filas = []
    detalles = {}

    for eje_cfg in ejes_final_por_tipo(tipo_real):
        eje_nombre = eje_cfg["nombre"]
        try:
            info_eje_base = estimar_info_eje_final(imagen, eje_cfg)
        except Exception as exc:
            filas.append({
                "split": split,
                "patient_id": patient_id,
                "tipo_real": tipo_real,
                "eje": eje_nombre,
                "escenario": f"{eje_nombre}_ERROR_EJE",
                "error": repr(exc),
            })
            continue

        for params in construir_grid_final(tipo_real, modo_rapido=modo_rapido):
            try:
                prompts_auto, info_eje, df_debug = generar_prompts_final_desde_info_eje(
                    imagen,
                    template_bbox_auto,
                    info_eje_base,
                    **params,
                )
                df_cajas = evaluar_cobertura_cajas_por_clase(prompts_auto, mascara)
                df_diag = diagnosticar_desplazamiento_cajas(prompts_auto, mascara)

                escenario = (
                    f"{eje_nombre}"
                    f"_oy{params['offset_y']}"
                    f"_sy{params['escala_pos_y']:.2f}"
                    f"_ml{params['mid_lift']}"
                    f"_gl{params['global_lift']}"
                    f"_tl{params['thoracic_lift']}"
                    f"_sl{params['mid_spine_lift']}"
                    f"_ll{params['lumbar_lift']}"
                    f"_w{params['escala_w']:.2f}"
                    f"_h{params['escala_h']:.2f}"
                )

                fila = {
                    "split": split,
                    "patient_id": patient_id,
                    "tipo_real": tipo_real,
                    "eje": eje_nombre,
                    "escenario": escenario,
                    "bbox_iou_promedio": df_cajas["bbox_iou"].mean(),
                    "bbox_recall_promedio": df_cajas["bbox_recall"].mean(),
                    "bbox_precision_promedio": df_cajas["bbox_precision"].mean(),
                    "bbox_iou_min": df_cajas["bbox_iou"].min(),
                    "n_vertebras_iou_mayor_02": int((df_cajas["bbox_iou"] >= 0.20).sum()),
                    "n_vertebras_recall_mayor_08": int((df_cajas["bbox_recall"] >= 0.80).sum()),
                    "dx_abs_promedio": df_diag["dx_box_gt"].abs().mean() if len(df_diag) else np.nan,
                    "dy_abs_promedio": df_diag["dy_box_gt"].abs().mean() if len(df_diag) else np.nan,
                    "dy_promedio_firmado": df_diag["dy_box_gt"].mean() if len(df_diag) else np.nan,
                    "error": "",
                    **params,
                }
                filas.append(fila)

                if fila["bbox_iou_promedio"] >= 0.30 or fila["n_vertebras_iou_mayor_02"] >= 16:
                    detalles[escenario] = {
                        "imagen": imagen,
                        "mascara": mascara,
                        "prompts": prompts_auto,
                        "info_eje": info_eje,
                        "df_cajas": df_cajas,
                        "df_diag": df_diag,
                        "df_debug": df_debug,
                    }

            except Exception as exc:
                filas.append({
                    "split": split,
                    "patient_id": patient_id,
                    "tipo_real": tipo_real,
                    "eje": eje_nombre,
                    "escenario": f"{eje_nombre}_ERROR_PARAMS",
                    "bbox_iou_promedio": np.nan,
                    "bbox_recall_promedio": np.nan,
                    "bbox_precision_promedio": np.nan,
                    "bbox_iou_min": np.nan,
                    "n_vertebras_iou_mayor_02": 0,
                    "n_vertebras_recall_mayor_08": 0,
                    "dx_abs_promedio": np.nan,
                    "dy_abs_promedio": np.nan,
                    "dy_promedio_firmado": np.nan,
                    "error": repr(exc),
                    **params,
                })

    df = pd.DataFrame(filas)
    if "bbox_iou_promedio" in df.columns:
        df = df.sort_values(
            ["bbox_iou_promedio", "n_vertebras_iou_mayor_02", "bbox_recall_promedio", "bbox_precision_promedio"],
            ascending=[False, False, False, False],
        ).reset_index(drop=True)

    return df, detalles


def correr_estrategia_final_split(split="val", patient_ids=None, modo_rapido=True, reanudar=True):
    if patient_ids is None:
        patient_ids = sorted(list(PROMPTS_DICC[split].keys()))

    checkpoint_csv = f"{PREFIJO_FINAL}.csv"
    resultados = []
    errores = []
    ya_hechos = set()

    if reanudar and Path(checkpoint_csv).exists():
        df_prev = pd.read_csv(checkpoint_csv)
        if not df_prev.empty and "patient_id" in df_prev.columns:
            resultados.append(df_prev)
            ya_hechos = set(df_prev["patient_id"].astype(str).unique())
            print(f"Reanudando {checkpoint_csv}: {len(ya_hechos)} pacientes ya estaban guardados.")

    pendientes = [p for p in patient_ids if str(p) not in ya_hechos]
    print(f"Pacientes pendientes: {len(pendientes)} / {len(patient_ids)}")
    print("Escenarios por normal:", len(EJES_FINAL_NORMAL) * len(construir_grid_final("normal", modo_rapido=modo_rapido)))
    print("Escenarios por escoliosis:", len(EJES_FINAL_ESCOLIOSIS) * len(construir_grid_final("escoliosis", modo_rapido=modo_rapido)))

    for patient_id in tqdm(pendientes, desc="vertebra semana final"):
        try:
            print(f"Procesando {patient_id} ({tipo_desde_patient_id(patient_id)})...")
            df_muestra, _ = evaluar_estrategia_final_muestra(split, patient_id, modo_rapido=modo_rapido)
            resultados.append(df_muestra)
            pd.concat(resultados, ignore_index=True).to_csv(checkpoint_csv, index=False)
        except Exception as exc:
            errores.append({"split": split, "patient_id": patient_id, "error": repr(exc)})
            pd.DataFrame(errores).to_csv(f"{PREFIJO_FINAL}_errores.csv", index=False)

    df_all = pd.concat(resultados, ignore_index=True) if resultados else pd.DataFrame()
    df_err = pd.DataFrame(errores)

    df_best = (
        df_all[df_all["error"].fillna("") == ""]
        .dropna(subset=["bbox_iou_promedio"])
        .sort_values(["patient_id", "bbox_iou_promedio", "n_vertebras_iou_mayor_02"], ascending=[True, False, False])
        .groupby("patient_id")
        .head(1)
        .reset_index(drop=True)
    ) if not df_all.empty else pd.DataFrame()

    df_decision = (
        df_all[df_all["error"].fillna("") == ""]
        .dropna(subset=["bbox_iou_promedio"])
        .groupby(["tipo_real", "eje"], as_index=False)
        .agg(
            pacientes=("patient_id", "nunique"),
            bbox_iou_promedio=("bbox_iou_promedio", "mean"),
            bbox_recall_promedio=("bbox_recall_promedio", "mean"),
            bbox_precision_promedio=("bbox_precision_promedio", "mean"),
            n_iou02=("n_vertebras_iou_mayor_02", "mean"),
            dy_abs_promedio=("dy_abs_promedio", "mean"),
        )
        .sort_values(["tipo_real", "bbox_iou_promedio"], ascending=[True, False])
    ) if not df_all.empty else pd.DataFrame()

    df_lifts = (
        df_all[df_all["error"].fillna("") == ""]
        .dropna(subset=["bbox_iou_promedio"])
        .groupby(["tipo_real", "global_lift", "thoracic_lift", "mid_spine_lift", "lumbar_lift"], as_index=False)
        .agg(
            escenarios=("escenario", "count"),
            bbox_iou_promedio=("bbox_iou_promedio", "mean"),
            dy_abs_promedio=("dy_abs_promedio", "mean"),
            dy_promedio_firmado=("dy_promedio_firmado", "mean"),
        )
        .sort_values(["tipo_real", "bbox_iou_promedio"], ascending=[True, False])
    ) if not df_all.empty else pd.DataFrame()

    df_top_params = (
        df_all[df_all["error"].fillna("") == ""]
        .dropna(subset=["bbox_iou_promedio"])
        .groupby(["tipo_real", "eje", "offset_y", "escala_pos_y", "mid_lift", "global_lift", "thoracic_lift", "mid_spine_lift", "lumbar_lift", "escala_w", "escala_h"], as_index=False)
        .agg(
            pacientes=("patient_id", "nunique"),
            bbox_iou_promedio=("bbox_iou_promedio", "mean"),
            bbox_recall_promedio=("bbox_recall_promedio", "mean"),
            bbox_precision_promedio=("bbox_precision_promedio", "mean"),
            dy_abs_promedio=("dy_abs_promedio", "mean"),
        )
        .sort_values(["tipo_real", "bbox_iou_promedio"], ascending=[True, False])
    ) if not df_all.empty else pd.DataFrame()

    df_all.to_csv(checkpoint_csv, index=False)
    df_best.to_csv(f"{PREFIJO_FINAL}_mejor_por_paciente.csv", index=False)
    df_decision.to_csv(f"{PREFIJO_FINAL}_decision_por_eje.csv", index=False)
    df_lifts.to_csv(f"{PREFIJO_FINAL}_efecto_lifts.csv", index=False)
    df_top_params.to_csv(f"{PREFIJO_FINAL}_top_parametros.csv", index=False)
    df_err.to_csv(f"{PREFIJO_FINAL}_errores.csv", index=False)

    return df_all, df_best, df_decision, df_lifts, df_err

def visualizar_mejor_final(split, patient_id, modo_rapido=True):
    df_muestra, detalles = evaluar_estrategia_final_muestra(split, patient_id, modo_rapido=modo_rapido)
    display(df_muestra.head(10))

    row = df_muestra.iloc[0]
    escenario = row["escenario"]
    if escenario in detalles:
        det = detalles[escenario]
    else:
        imagen, mascara, prompts_auto, info_eje, _ = prompts_desde_fila_final(row)
        det = {"imagen": imagen, "mascara": mascara, "prompts": prompts_auto, "info_eje": info_eje}

    visualizar_cajas_auto_bordes(
        img_rgb=det["imagen"],
        prompts_auto=det["prompts"],
        info_eje=det["info_eje"],
        mascara_gt=det["mascara"],
        titulo=(
            f"{patient_id} | {row['escenario']} | "
            f"IoU={row['bbox_iou_promedio']:.3f} | "
            f"recall={row['bbox_recall_promedio']:.3f} | "
            f"dy={row['dy_abs_promedio']:.1f}"
        ),
    )
    return df_muestra, detalles


def prompts_desde_fila_final(row):
    split = row["split"]
    patient_id = row["patient_id"]
    tipo_real = row.get("tipo_real", tipo_desde_patient_id(patient_id))
    path_img, path_mask = resolver_paths_muestra(split, patient_id)
    imagen = cargar_imagen(path_img)
    mascara = cargar_mascara(path_mask)

    eje_cfg = next(e for e in ejes_final_por_tipo(tipo_real) if e["nombre"] == row["eje"])
    info_eje_base = estimar_info_eje_final(imagen, eje_cfg)
    params = {
        "escala_w": float(row["escala_w"]),
        "escala_h": float(row["escala_h"]),
        "peso_eje": float(row.get("peso_eje", 1.0)),
        "escala_pos_y": float(row["escala_pos_y"]),
        "offset_y": float(row["offset_y"]),
        "offset_x": float(row.get("offset_x", -10)),
        "mid_lift": float(row["mid_lift"]),
        "global_lift": float(row.get("global_lift", 0)),
        "thoracic_lift": float(row.get("thoracic_lift", 0)),
        "mid_spine_lift": float(row.get("mid_spine_lift", 0)),
        "lumbar_lift": float(row.get("lumbar_lift", 0)),
    }
    prompts_auto, info_eje, df_debug = generar_prompts_final_desde_info_eje(
        imagen,
        template_bbox_auto,
        info_eje_base,
        **params,
    )
    return imagen, mascara, prompts_auto, info_eje, df_debug


def evaluar_medsam_basico_con_mejores(df_best, max_pacientes=6):
    if predictor is None:
        raise RuntimeError("Primero activa EJECUTAR_CARGA_MEDSAM=True y ejecuta la celda de MedSAM.")

    resultados = []
    for _, row in tqdm(df_best.head(max_pacientes).iterrows(), total=min(len(df_best), max_pacientes), desc="MedSAM basico"):
        imagen, mascara, prompts_auto, info_eje, _ = prompts_desde_fila_final(row)
        mask_sem_pred, mask_sem_gt, detalles, score_map, quality = reconstruir_mascara_semantica_medsam_con_score(
            imagen=imagen,
            mascara_gt_multiclase=mascara,
            prompts_sample=prompts_auto,
            predictor=predictor,
            frac_x=0.04,
            frac_y=0.06,
            return_quality=True,
        )
        resultados.append({
            "split": row["split"],
            "patient_id": row["patient_id"],
            "tipo_real": row["tipo_real"],
            "eje": row["eje"],
            "escenario": row["escenario"],
            "dice_macro": dice_multiclase_promedio(mask_sem_pred, mask_sem_gt, list(range(1, N_CLASES + 1))),
            "iou_macro": iou_multiclase_promedio(mask_sem_pred, mask_sem_gt, list(range(1, N_CLASES + 1))),
            "overlap_fraction": quality["overlap_fraction_pred"],
            "pred_vacias": quality["n_predicciones_vacias"],
        })

    df_res = pd.DataFrame(resultados)
    df_res.to_csv(f"{PREFIJO_FINAL}_medsam_basico.csv", index=False)
    return df_res


if EJECUTAR_PRUEBA_FINAL_CORTA:
    df_final, df_final_best, df_final_decision, df_final_lifts, df_final_err = correr_estrategia_final_split(
        split="val",
        modo_rapido=True,
    )
    display(df_final_best)
    display(df_final_decision)
    display(df_final_lifts.groupby("tipo_real").head(12))
    display(df_final_err)

if EJECUTAR_VISUAL_FINAL:
    for pid in PACIENTES_VISUALES_FINAL:
        if pid in PROMPTS_DICC["val"]:
            visualizar_mejor_final("val", pid, modo_rapido=True)

if EJECUTAR_MEDSAM_BASICO:
    if "df_final_best" not in globals():
        df_final_best = pd.read_csv(f"{PREFIJO_FINAL}_mejor_por_paciente.csv")
    df_medsam_basico = evaluar_medsam_basico_con_mejores(df_final_best, max_pacientes=6)
    display(df_medsam_basico)